# Memory in LangChain Agents

This reference notebook compares two agent configurations:

- **No memory:** each invocation receives only the message supplied in that call.
- **Checkpointed memory:** an `InMemorySaver` stores the conversation state for a thread, allowing later invocations with the same `thread_id` to use earlier messages.

## Learning goals

- See why an agent does not automatically remember previous calls.
- Configure an in-memory checkpointer for a development example.
- Use a stable `thread_id` when invoking a checkpointed agent.
- Understand the difference between short-term thread memory and durable production storage.

## Before you run the notebook

1. Load a model-provider API key through the project's `.env` file or environment.
2. Run the cells from top to bottom.
3. The `InMemorySaver` example stores data only while the current Python process is running; it is intended for learning and testing, not durable production memory.

In [1]:
# Load API keys and other local settings from the project's .env file.
from dotenv import load_dotenv

load_dotenv()

# This reference notebook runs offline by default, so prevent optional
# LangSmith callbacks from attempting network uploads.
import os

os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

for environment_variable in ("LANGSMITH_API_KEY", "LANGCHAIN_API_KEY"):
    os.environ.pop(environment_variable, None)

# Restart the notebook kernel after changing this cell so old callbacks unload.
# Set LANGSMITH_TRACING=true and provide a LangSmith API key manually when tracing is needed.

## 1. No Memory

This section intentionally creates a stateless agent. Each call below sends one new message, so the second call does not include the first call's conversation history.

In [2]:
from langchain.agents import create_agent

# Without a checkpointer, the agent has no persisted conversation state.
agent = create_agent(model="gpt-5-nano")

In [3]:
from langchain.messages import HumanMessage

# This message contains the user's name and favorite color for this call only.
question = HumanMessage(
    content="Hello, my name is Michael and my favorite color is blue."
)

# The messages list is the complete conversation supplied to this invocation.
response = agent.invoke({"messages": [question]})

In [4]:
from pprint import pprint

# Inspect the full state returned by the agent, including its message history.
pprint(response)

{'messages': [HumanMessage(content='Hello, my name is Michael and my favorite color is blue.', additional_kwargs={}, response_metadata={}, id='aba78595-3578-4346-a658-594a3a38c66f'),
              AIMessage(content='Nice to meet you, Michael. Blue is a great favorite—calming and versatile. What would you like to do today? We can chat about blue (color ideas, design tips, symbolism), or we can switch to any other topic you’re interested in.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 702, 'prompt_tokens': 19, 'total_tokens': 721, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EPouFd7ccAweGo9ggQnguc6Z1O8h1', 'service_tier': 'default', 'finish_

In [5]:
# This is a new invocation with no prior messages included.
question = HumanMessage(content="What is my favorite color?")

response = agent.invoke({"messages": [question]})

# The model may say it does not know because the earlier message was not supplied.
pprint(response)

{'messages': [HumanMessage(content='What is my favorite color?', additional_kwargs={}, response_metadata={}, id='48579611-9925-4826-9abc-f822c758d5fa'),
              AIMessage(content='I don’t know your favorite color yet. Do you want to tell me, or should we do a quick quiz to figure it out?\n\nIf you’d like a quick guide (no questions needed):\n- Calm vibe: blue or green\n- Energetic vibe: red or orange\n- Elegant/creative vibe: purple\n- Minimal/sleek vibe: black, white, or gray\n\nIf you’d rather do a short quiz, answer these with A/B/C/D and I’ll guess:\n1) Which vibe do you want the color to give? A) calm and cool (blue/green) B) bold and energetic (red/orange) C) elegant/creative (purple) D) clean/minimal (black/white/gray)\n2) Which do you notice first in a palette? A) blue B) red C) green D) purple\n3) For a room or outfit, which mood do you want? A) tranquil B) lively C) cozy/natural D) dramatic\n4) Pick an accent style: A) teal/yellow B) hot pink/orange C) earthy tones D) b

### No-memory summary

A model can only use the messages provided for the current invocation. Creating an agent without a checkpointer does not create durable conversation memory, so the second question cannot reliably use the first statement.

## 2. Short-Term Thread Memory

A checkpointer saves the agent state between invocations. The state is associated with a `thread_id`, so every call that should share memory must use the same identifier.

In [6]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver keeps checkpointed state in process memory.
# Use a durable checkpointer when state must survive restarts or be shared.
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
)

In [7]:
from langchain.messages import HumanMessage

question = HumanMessage(
    content="Hello, my name is Michael and my favorite color is blue."
)

# All calls in this conversation must use the same thread_id.
config = {"configurable": {"thread_id": "memory-demo-1"}}

# The checkpointer stores this message under the selected thread.
response = agent.invoke({"messages": [question]}, config)

In [8]:
# Inspect the checkpointed state after the first turn.
pprint(response)

{'messages': [HumanMessage(content='Hello, my name is Michael and my favorite color is blue.', additional_kwargs={}, response_metadata={}, id='71e526df-1458-4deb-b076-ab63c588673d'),
              AIMessage(content='Hi Michael! Nice to meet you. Blue is a great choice—calming and versatile. What would you like to chat about today? I can help with color palettes, design ideas, or anything else you’re interested in.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 375, 'prompt_tokens': 19, 'total_tokens': 394, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EPouUcNvqT5SiIbfCa7tP4JiaUCGJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs

In [9]:
question = HumanMessage(content="What's my favorite color?")

# Reusing the same config lets the agent retrieve the earlier turn.
response = agent.invoke({"messages": [question]}, config)

pprint(response)

{'messages': [HumanMessage(content='Hello, my name is Michael and my favorite color is blue.', additional_kwargs={}, response_metadata={}, id='71e526df-1458-4deb-b076-ab63c588673d'),
              AIMessage(content='Hi Michael! Nice to meet you. Blue is a great choice—calming and versatile. What would you like to chat about today? I can help with color palettes, design ideas, or anything else you’re interested in.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 375, 'prompt_tokens': 19, 'total_tokens': 394, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EPouUcNvqT5SiIbfCa7tP4JiaUCGJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs

### Memory section summary

`InMemorySaver` enables short-term memory by saving state for each thread. The second invocation can answer the color question because it uses the same `thread_id` and therefore retrieves the first turn's messages.

## Conclusion and reference checklist

The key distinction is where conversation state comes from:

- Without a checkpointer, include the complete conversation history in each request yourself.
- With a checkpointer, pass the same `thread_id` so LangGraph can restore the saved state.
- Use `InMemorySaver` for local experiments and tests. Choose a durable checkpointer for applications that need state after a process restart.
- Treat a thread as a conversation boundary: use a new identifier when a user starts a separate conversation.

The reusable invocation pattern is `agent.invoke({"messages": [message]}, config)` for checkpointed conversations, and `agent.invoke({"messages": [message]})` for stateless calls.